# Mastering Python Concurrency: Free-Threading & Sub-Interpreters

A comprehensive learning notebook covering the Globomantics Risk Engine course.

## Modules Covered
1. Understanding the GIL — Why threads don't scale
2. Mixing asyncio with CPU-bound work
3. Running Python without the GIL (free-threading, PEP 703)
4. Isolating parallel work with sub-interpreters
5. Profiling free-threaded and sub-interpreter workloads
6. Shipping free-threaded Python to production

**Running example throughout:** *Globomantics Risk Engine* — a CPU-bound Monte Carlo portfolio-risk simulator that runs thousands of P&L scenarios to compute Value-at-Risk (VaR).

---
## Table of Contents
- [Module 1 — Understanding the GIL](#module-1)
- [Module 2 — asyncio + CPU-bound Work](#module-2)
- [Module 3 — Running Without the GIL](#module-3)
- [Module 4 — Sub-Interpreters](#module-4)
- [Module 5 — Profiling Parallel Workloads](#module-5)
- [Module 6 — Shipping to Production](#module-6)

---
<a id='module-1'></a>
# Module 1 — Understanding the GIL: Why Threads Don't Scale

## 1.1 What Is the GIL?

The **Global Interpreter Lock (GIL)** is a single process-wide mutex inside CPython. Only one thread may hold the GIL at a time, which means:

- **Threads are concurrent, never parallel** for pure-Python bytecode.
- The scheduler switches threads roughly every **5 ms** (`sys.getswitchinterval()`).
- Multiple threads take turns — they never run simultaneously on separate CPU cores.

### Why does the GIL exist?

Every Python object has a **reference count** (`ob_refcnt`). The GIL protects that counter:

```
ob_refcnt += 1   ← load / add / store — three separate machine instructions
```

Without the GIL two threads could race on that three-step update and corrupt memory. The GIL is the simplest way to make reference counting safe without per-object synchronisation.

### Python concurrency timeline

| Era | Feature | Python version |
|-----|---------|----------------|
| Early 1990s | `threading` module | 1.x |
| 2008 | `multiprocessing` | 2.6 |
| 2012 | `concurrent.futures` | 3.2 |
| 2014 | `asyncio` | 3.4 |
| 2023 | PEP 703 (free-threading opt-in) | 3.13 |
| 2025 | Free-threading + sub-interpreters standard | 3.14 |

In [1]:
import sys

print(f"Python: {sys.version}")
print(f"Switch interval: {sys.getswitchinterval() * 1000:.0f} ms")
print(f"GIL enabled: {sys._is_gil_enabled()}")

Python: 3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 21.0.0 (clang-2100.0.123.102)]
Switch interval: 5 ms
GIL enabled: True


## 1.2 CPU-bound vs IO-bound Under the GIL

| Workload type | GIL behaviour | Threads help? |
|--------------|---------------|---------------|
| **CPU-bound Python** | Thread holds GIL the entire bytecode slice | ❌ Threads serialize; no speedup |
| **IO-bound** (sockets, files) | Thread releases GIL while waiting for the OS | ✅ Other threads run during IO |
| **C extensions** (NumPy, etc.) | Extension can release GIL during heavy math | ✅ Parallel math is possible |

The Globomantics risk engine loops over Monte Carlo scenarios in pure Python — it is **CPU-bound** and the GIL prevents thread-level parallelism.

In [2]:
# The core simulate.py from Module 1
import math
import random
from dataclasses import dataclass

HORIZON_DAYS = 30
TRADING_DAYS = 252


@dataclass(frozen=True)
class Position:
    symbol: str
    spot: float       # current price
    volatility: float # annualised vol
    quantity: int     # shares held


def _simulate_position_pnl(
    position: Position, scenarios: int, rng: random.Random
) -> list[float]:
    """Monte Carlo P&L for a single position across N scenarios."""
    daily_vol = position.volatility / math.sqrt(TRADING_DAYS)
    pnls = []
    for _ in range(scenarios):
        price = position.spot
        for _ in range(HORIZON_DAYS):
            price *= 1.0 + rng.gauss(0.0, daily_vol)
        pnls.append((price - position.spot) * position.quantity)
    return pnls


def run_scenarios(
    positions: list[Position], scenarios: int, seed: int
) -> list[float]:
    """Run Monte Carlo across all positions; return per-scenario totals."""
    rng = random.Random(seed)
    totals = [0.0] * scenarios
    for position in positions:
        for i, pnl in enumerate(_simulate_position_pnl(position, scenarios, rng)):
            totals[i] += pnl
    return totals


def summarize(pnls: list[float]) -> dict[str, float]:
    """Compute VaR-95, Expected Shortfall, and mean."""
    ordered = sorted(pnls)
    cutoff = max(1, len(ordered) // 20)
    tail = ordered[:cutoff]
    return {
        "var95": -ordered[cutoff - 1],
        "es95": -sum(tail) / len(tail),
        "mean": sum(pnls) / len(pnls),
    }


# --- demo portfolio ---
def demo_portfolio() -> list[Position]:
    return [
        Position("AAPL",  182.0, 0.28, 100),
        Position("MSFT",  370.0, 0.24,  80),
        Position("NVDA",  495.0, 0.55,  50),
        Position("TSLA",  215.0, 0.62,  60),
    ]


# --- quick test ---
positions = demo_portfolio()
pnls = run_scenarios(positions, scenarios=1_000, seed=7)
stats = summarize(pnls)
print(f"VaR-95  : ${stats['var95']:,.2f}")
print(f"ES-95   : ${stats['es95']:,.2f}")
print(f"Mean P&L: ${stats['mean']:,.2f}")

VaR-95  : $9,539.45
ES-95   : $11,363.80
Mean P&L: $246.81


## 1.3 Proving the GIL Bottleneck

The benchmark compares three parallelism strategies on the CPU-bound simulation:

1. **Single-threaded** — baseline
2. **Threads** (1 / 2 / 4 / 8) — GIL forces serialisation, no speedup expected
3. **Processes** (8) — each process has its own GIL; real parallelism

Expected output on a standard CPython build:

```
mode         workers     wall  speedup        var95
single             1    1.41s    1.00x    71,618.80
threads            1    1.39s    1.01x    71,618.80
threads            2    2.11s    0.67x    71,618.80  ← SLOWER (lock contention)
threads            4    2.34s    0.60x    71,618.80
threads            8    2.60s    0.54x    71,618.80
processes          8    0.42s    3.36x    71,618.80  ← real parallelism
```

**Key insight:** threads slow down the CPU-bound workload. Processes use separate GILs and scale.

In [3]:
import threading
import time
from concurrent.futures import ProcessPoolExecutor


def _chunk_sizes(total: int, workers: int) -> list[int]:
    base, extra = divmod(total, workers)
    return [base + (1 if i < extra else 0) for i in range(workers)]


def run_threads(
    positions: list[Position], scenarios: int, workers: int, seed: int
) -> list[float]:
    sizes = _chunk_sizes(scenarios, workers)
    results: list[list[float]] = [[] for _ in sizes]

    def worker(index: int, count: int) -> None:
        results[index] = run_scenarios(positions, count, seed + index)

    threads = [
        threading.Thread(target=worker, args=(i, count))
        for i, count in enumerate(sizes)
    ]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return [pnl for chunk in results for pnl in chunk]


# NOTE: run_processes requires an importable module; shown here for illustration.
# In the course demos it is called as:
#   from concurrent.futures import ProcessPoolExecutor
#   with ProcessPoolExecutor(max_workers=8) as pool:
#       futures = [pool.submit(run_scenarios, positions, count, seed+i) ...]

# --- mini benchmark (threads only, safe in a notebook) ---
positions = demo_portfolio()
SCENARIOS = 4_000
SEED = 7

start = time.perf_counter()
pnls_single = run_scenarios(positions, SCENARIOS, SEED)
t_single = time.perf_counter() - start

start = time.perf_counter()
pnls_8t = run_threads(positions, SCENARIOS, 8, SEED)
t_8threads = time.perf_counter() - start

print(f"Single-threaded : {t_single:.3f}s")
print(f"8 threads       : {t_8threads:.3f}s  (speedup: {t_single/t_8threads:.2f}x)")
print("\nNote: on a standard GIL build, 8 threads are often SLOWER than 1.")

Single-threaded : 0.048s
8 threads       : 0.049s  (speedup: 0.98x)

Note: on a standard GIL build, 8 threads are often SLOWER than 1.


## 1.4 Multiprocessing: Real Parallelism at a Cost

Processes bypass the GIL because each gets **its own interpreter instance**. The cost:

| Cost | Details |
|------|---------|
| Spawn time | ~300 ms per process (platform-dependent) |
| Pickle overhead | All function args and return values are serialised |
| Memory | Each worker loads a full copy of your modules |
| IPC | OS-level inter-process communication required |

For a 40,000-scenario job the spawn cost is amortised — processes win. For tiny jobs, spawn dominates and processes can be slower than single-threaded.

## Module 1 — Key Takeaways

- The GIL is a process-wide mutex protecting reference counting — not a design flaw, but an intentional simplicity trade-off.
- CPU-bound Python threads **serialize**, never parallelize. Adding threads to a CPU-bound loop often makes it slower.
- IO-bound and C-extension work can release the GIL and benefit from threads.
- `multiprocessing` achieves real parallelism by spawning separate GIL-holding processes — but pays spawn and pickle costs.

---
<a id='module-2'></a>
# Module 2 — Mixing asyncio with CPU-bound Work

## 2.1 The asyncio Execution Model

asyncio runs on **one thread** with **one event loop**. Execution is **cooperative** (not preemptive):

```
coroutine A runs ──► await something ──► yields back to event loop
event loop ─────────────────────────────► runs coroutine B
...
event loop ─────────────────────────────► resumes A when its awaitable is ready
```

asyncio is the right front door for **IO fan-out** — accepting hundreds of network connections and dispatching them cheaply.

In [4]:
import asyncio

async def demo_cooperative():
    """Demonstrates cooperative scheduling: tasks yield at await points."""
    print("Task A: starting")
    await asyncio.sleep(0)  # yields to event loop
    print("Task A: resumed after yield")

async def demo_tasks():
    async with asyncio.TaskGroup() as tg:
        tg.create_task(demo_cooperative())
        tg.create_task(demo_cooperative())

await demo_tasks()

Task A: starting
Task A: starting
Task A: resumed after yield
Task A: resumed after yield


## 2.2 Event-Loop Starvation

If a coroutine **never awaits**, it holds the event loop forever — all other coroutines are starved.

```python
async def bad_handler(job):
    # THIS BLOCKS THE ENTIRE EVENT LOOP
    result = run_scenarios(positions, 40_000, seed)  # CPU-bound, no await
    return result
```

While `run_scenarios` runs:
- Health-check requests pile up unanswered
- Incoming connections queue up
- A single 7-second simulation blocks everything

**Measurement:** one stalled connection can show up as p99 latency 7+ seconds even when the mean is < 0.1 s.

## 2.3 The asyncio Hybrid Boundary

The correct pattern: let the **event loop own IO**, offload **CPU to a thread pool** using `asyncio.to_thread`.

```
Client ──► asyncio event loop ──► await fetch_quotes()   (IO: asyncio)
                               └─► await asyncio.to_thread(run_job, ...)  (CPU: thread)
```

`asyncio.to_thread` submits a blocking callable to the default `ThreadPoolExecutor` and returns an awaitable. The event loop remains free to serve other connections while the CPU work runs in the background.

In [5]:
import asyncio
import time


def cpu_bound_job(n: int) -> float:
    """Simulates a CPU-bound task (blocking)."""
    return sum(i * i for i in range(n))


async def handler(job_id: int, n: int):
    print(f"[{job_id}] Starting IO simulation...")
    await asyncio.sleep(0.05)  # simulate IO (network, DB, etc.)
    print(f"[{job_id}] IO done. Offloading CPU work to thread pool...")

    # KEY PATTERN: asyncio.to_thread keeps the event loop free
    result = await asyncio.to_thread(cpu_bound_job, n)
    print(f"[{job_id}] CPU done. result={result}")
    return result


async def main():
    start = time.perf_counter()
    # Both handlers run concurrently — IO interleaves, CPU uses threads
    async with asyncio.TaskGroup() as tg:
        tg.create_task(handler(1, 1_000_000))
        tg.create_task(handler(2, 1_000_000))
    print(f"\nTotal wall time: {time.perf_counter() - start:.3f}s")


await main()

[1] Starting IO simulation...
[2] Starting IO simulation...
[1] IO done. Offloading CPU work to thread pool...
[2] IO done. Offloading CPU work to thread pool...
[1] CPU done. result=333332833333500000
[2] CPU done. result=333332833333500000

Total wall time: 0.095s


## 2.4 The Risk Engine Async Server Pattern

From the course demo (`server.py`, Module 2):

In [6]:
# Pattern from: 02/demos/3. Offloading CPU work off the event loop/risk_engine/server.py
# Shown here for study — not run in notebook (requires running server)

ASYNC_SERVER_PATTERN = '''
async def handle_simulate(job: dict) -> dict:
    # 1. Async IO: fetch live market quotes (event loop stays free)
    quotes = await fetch_quotes(symbols)

    # 2. CPU offload: run Monte Carlo in a thread (event loop stays free)
    report = await asyncio.to_thread(run_job, scenarios, seed, quotes)

    return {
        "var95": report["var95"],
        "gil_enabled": sys._is_gil_enabled(),   # always True on standard build
        ...
    }
'''
print(ASYNC_SERVER_PATTERN)


async def handle_simulate(job: dict) -> dict:
    # 1. Async IO: fetch live market quotes (event loop stays free)
    quotes = await fetch_quotes(symbols)

    # 2. CPU offload: run Monte Carlo in a thread (event loop stays free)
    report = await asyncio.to_thread(run_job, scenarios, seed, quotes)

    return {
        "var95": report["var95"],
        "gil_enabled": sys._is_gil_enabled(),   # always True on standard build
        ...
    }



## 2.5 Honest Cost Sheet for asyncio

| Problem | Does asyncio solve it? |
|---------|------------------------|
| Hundreds of simultaneous IO connections | ✅ Yes |
| CPU-bound throughput | ❌ No (single thread) |
| Event-loop starvation from CPU work | ✅ Solved with `asyncio.to_thread` |
| True CPU parallelism | ❌ Need threads (free-threaded) or processes |

asyncio **solves latency** (one slow job no longer blocks others), but does **not increase CPU throughput** — `to_thread` still uses the GIL.

## Module 2 — Key Takeaways

- asyncio is single-threaded and cooperative: a coroutine runs until it `await`s.
- Inline CPU work **starves** the event loop — never block inside an `async def`.
- `asyncio.to_thread(fn, *args)` moves blocking work off the event loop; it is the correct async/CPU boundary.
- asyncio + `to_thread` solves **latency tail** but not **CPU throughput** — the GIL still limits parallelism.

---
<a id='module-3'></a>
# Module 3 — Running Python Without the GIL (Free-Threading)

## 3.1 PEP 703: Free-Threaded CPython

**PEP 703** removes the GIL from CPython. Key facts:

- Introduced as experimental opt-in in **Python 3.13**.
- Becomes the standard (two co-equal builds) in **Python 3.14**.
- Same language, same imports, same code — only the runtime differs.
- Install the free-threaded build: `uv python install 3.14t`

### Two binaries in 3.14
```
python3.14   ← standard build  (GIL on)
python3.14t  ← free-threaded   (GIL removed)
```

In [7]:
import sys
import sysconfig


def build_kind() -> str:
    if sysconfig.get_config_var("Py_GIL_DISABLED"):
        return "free-threaded"
    return "standard"


def narrate(build: str, gil_enabled: bool) -> str:
    if build == "standard":
        return "GIL is compiled in; switching builds is the only way off"
    if gil_enabled:
        return "GIL was forced back on for this run (-X gil=1 or incompatible extension)"
    return "threads can run CPU-bound code in parallel!"


build = build_kind()
gil_on = sys._is_gil_enabled()
print(f"Python {sys.version.split()[0]}")
print(f"Build: {build}")
print(f"GIL enabled: {gil_on}")
print(f"Status: {narrate(build, gil_on)}")

Python 3.14.6
Build: standard
GIL enabled: True
Status: GIL is compiled in; switching builds is the only way off


## 3.2 How Free-Threading Makes Memory Safe

Without the GIL, concurrent reference-count updates would corrupt memory. Three mechanisms replace it:

### Per-object locking
Each mutable Python object has a lightweight per-object lock. Threads only contend when they touch the **same** object simultaneously — unrelated objects are fully independent.

### Biased reference counting
Each object has two counters:
- **Local count** — for the owning thread (no atomics, cheap)
- **Shared count** — for other threads (atomic operations)

Most objects are created and destroyed on a single thread, so the local path dominates.

### Deferred reference counting
Modules, classes, and functions defer their counts. **Immortal objects** (None, True, False, small integers) are never counted at all — zero cost.

### Result: single-thread overhead
```
Standard build  ~1.41 s / 40k scenarios
Free-threaded   ~1.70 s / 40k scenarios   (~20% slower single-thread)
```
At 8 threads the free-threaded build delivers **~3.7× speedup** — turning the 20% overhead into a net win.

In [8]:
# On a free-threaded build (python3.14t), this achieves REAL parallelism:
# threads run simultaneously on different CPU cores.
#
# On a standard build, run_threads() above will serialize (GIL prevents parallelism).
# The CODE is identical — only the interpreter binary differs.

import threading
import time


def worker_sum(n: int, results: list, idx: int) -> None:
    results[idx] = sum(i * i for i in range(n))


def benchmark_threads(n: int, num_threads: int) -> float:
    results = [0] * num_threads
    threads = [
        threading.Thread(target=worker_sum, args=(n, results, i))
        for i in range(num_threads)
    ]
    start = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return time.perf_counter() - start


N = 500_000
t1 = benchmark_threads(N, 1)
t4 = benchmark_threads(N, 4)
print(f"1 thread : {t1:.3f}s")
print(f"4 threads: {t4:.3f}s  (speedup: {t1/t4:.2f}x)")
print(f"\nGIL enabled: {sys._is_gil_enabled()}")
print("On free-threaded build, 4 threads should be ~3-4x faster.")

1 thread : 0.011s
4 threads: 0.041s  (speedup: 0.27x)

GIL enabled: True
On free-threaded build, 4 threads should be ~3-4x faster.


## 3.3 C-Extension Compatibility

C extensions declare free-thread safety via the `Py_mod_gil` slot:

```c
// In a C extension:
static PyModuleDef_Slot slots[] = {
    {Py_mod_gil, Py_MOD_GIL_NOT_USED},  // I am thread-safe
    {0, NULL}
};
```

**Important:** if any import in your process loads an **incompatible** extension, CPython **silently re-enables the GIL** at runtime with a warning:
```
RuntimeWarning: the GIL was re-enabled to load extension mymodule
```

Check GIL status after startup:
```python
assert not sys._is_gil_enabled(), "GIL re-enabled — check extension compatibility"
```

### Wheels on PyPI
Free-threaded wheels for `cp314t` are being added. The adoption is growing rapidly.

## 3.4 Thread-Safety Red Zones

On a free-threaded build, **races that were impossible under the GIL now happen**. The four red zones:

| Red zone | Example | Fix |
|----------|---------|-----|
| Shared mutable globals | Module-level cache dict | `threading.Lock` or per-thread state |
| Lazy-initialized caches | `if cache is None: cache = ...` | Lock the init block |
| Compound updates | `total += batch` (load → add → store) | Lock or use `partial` pattern |
| Multi-field invariants | `count` and `total` must stay in sync | Lock the entire multi-field update |

In [9]:
# Demonstration of the race condition in a shared aggregator
# (From Module 3 demo: risk_engine/aggregator.py)

import threading


class RacyCounter:
    """Unsafe: compound update (load/add/store) is not atomic."""
    def __init__(self): self.count = 0
    def increment(self): self.count += 1   # ← race on free-threaded build


class SafeCounter:
    """Safe: lock protects the compound update."""
    def __init__(self):
        self.count = 0
        self._lock = threading.Lock()
    def increment(self):
        with self._lock:
            self.count += 1


class PartialCounter:
    """Best pattern: each thread accumulates locally, merge once at the end."""
    def __init__(self): self.count = 0
    def increment(self): self.count += 1   # only one thread touches this instance


def stress_counter(counter, num_threads=8, increments=10_000):
    barrier = threading.Barrier(num_threads)
    def run():
        barrier.wait()  # all threads start simultaneously
        for _ in range(increments):
            counter.increment()
    threads = [threading.Thread(target=run) for _ in range(num_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return counter.count


expected = 8 * 10_000
result_safe = stress_counter(SafeCounter())
print(f"Expected : {expected:,}")
print(f"Safe     : {result_safe:,}  match={result_safe == expected}")
# RacyCounter may show corruption on a free-threaded build;
# on standard GIL build it appears correct (GIL hides the race)

Expected : 80,000
Safe     : 80,000  match=True


## 3.5 The Partial-Aggregation Pattern (Lock-Free)

Instead of a shared locked aggregator, give each thread **its own private partial accumulator** and merge at the end — on the event-loop thread:

In [10]:
from concurrent.futures import ThreadPoolExecutor


def fold_partial(worker_id: int, batches: int) -> dict:
    """Each thread accumulates locally — zero shared state during computation."""
    total = 0
    for _ in range(batches):
        total += 1  # simulates accumulating results
    return {"worker": worker_id, "total": total}


def run_partials(workers: int, batches_per_worker: int) -> int:
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [
            pool.submit(fold_partial, i, batches_per_worker)
            for i in range(workers)
        ]
        partials = [f.result() for f in futures]

    # Merge is single-threaded — safe by construction
    grand_total = sum(p["total"] for p in partials)
    return grand_total


result = run_partials(workers=8, batches_per_worker=10_000)
expected = 8 * 10_000
print(f"Expected: {expected:,}")
print(f"Actual  : {result:,}")
print(f"Match   : {result == expected}")
print("\nPartial pattern: zero lock contention, always correct.")

Expected: 80,000
Actual  : 80,000
Match   : True

Partial pattern: zero lock contention, always correct.


## 3.6 Detecting Races Before They Ship

Three techniques from the course:

1. **ThreadSanitizer (TSan)** — compile-time instrumentation that detects races at runtime. Requires a special Python build.

2. **Stress tests** — run with high thread count + many iterations + a barrier so all threads start simultaneously. The barrier amplifies race windows.

3. **Invariant checks** — after parallel work, verify that `observed == expected` (like the course's `verdict=CORRUPTED/CLEAN` check).

## 3.7 Free-Threading Adoption Checklist

Before enabling free-threading in production:

1. **C-extension audit** — check all imports for `cp314t` wheels or `Py_mod_gil` declarations.
2. **Global-state audit** — find every module-level mutable dict/list/counter.
3. **Test obligations** — run your full test suite on **both** standard and free-threaded builds (dual-build CI).
4. **Benchmarking obligations** — measure single-thread overhead (~20%) against multi-thread speedup.

The goal of the audit: prove "the hot path shares nothing." If it does, there are zero code changes to scale.

## Module 3 — Key Takeaways

- Free-threading (PEP 703) removes the GIL; threads achieve true CPU parallelism.
- Three safety mechanisms replace the GIL: per-object locks, biased ref counting, deferred ref counting.
- Single-thread overhead is ~20%; 8-thread speedup is ~3.7× on the risk engine.
- One incompatible C extension silently re-enables the GIL — always assert `not sys._is_gil_enabled()`.
- Free-threaded build exposes races that the GIL was hiding — audit globals, use partial aggregation.

---
<a id='module-4'></a>
# Module 4 — Isolating Parallel Work with Sub-Interpreters

## 4.1 Two Safety Models for Parallel Python

| | Free-threading | Sub-interpreters |
|-|---------------|------------------|
| **Sharing** | Threads share everything by default | Share nothing by default |
| **Safety** | Discipline: locks, partials, stress tests | Construction: isolation enforced by runtime |
| **GIL** | One GIL removed | One private GIL per interpreter |
| **Build required** | `python3.14t` (free-threaded) | `python3.14` (standard build) |
| **Single-thread cost** | ~20% overhead | None |

## 4.2 The Sub-Interpreter Model

A sub-interpreter is a **complete, independent Python world** inside a single process:
- Own modules, own globals, own `sys.path`, **own GIL**
- Carried by threads — no spawn, no OS-level IPC
- 8 interpreters on 8 threads → 8 private GILs → **CPU-bound code runs genuinely in parallel** on the **standard build**

### PEPs that enable this
- **PEP 684** (3.12) — private GIL per interpreter
- **PEP 734** (3.14) — creation and control in the standard library (`concurrent.interpreters`)

## 4.3 Threads vs Sub-Interpreters vs Processes

| | Threads | Sub-interpreters | Processes |
|-|---------|-----------------|----------|
| Memory | One shared heap | Process-like isolation | Full isolation |
| Startup | Instant | Close to thread-like | ~300 ms spawn |
| IPC cost | None (shared memory) | Pickle across boundary | Pickle on every crossing |
| Race risk | High (everything shared) | Low (nothing shared) | None |
| GIL | One (blocks CPU) | One per interpreter | One per process |
| Standard build | ✅ | ✅ | ✅ |
| Free-thread build | ✅ better | ✅ still works | ✅ |

## 4.4 Two Levels of the Standard-Library API

### Level 1: `InterpreterPoolExecutor` (high-level, easy)
A `concurrent.futures` executor — the third sibling of `ThreadPoolExecutor` and `ProcessPoolExecutor`. Every worker is its own interpreter.

### Level 2: `concurrent.interpreters` (low-level, flexible)
Create interpreters directly and wire up queues across the isolation boundary by hand.

In [11]:
# From: 04/demos/2. Running simulations in sub-interpreters/risk_engine/parallel_interp.py
# Requires Python 3.14+ with concurrent.futures.InterpreterPoolExecutor

# Shown as a code study cell — run on Python 3.14

SUB_INTERP_PATTERN = '''
from concurrent.futures import InterpreterPoolExecutor

def _interp_worker(
    positions: list[Position], count: int, seed: int
) -> list[float]:
    # This function runs in its own sub-interpreter.
    # It imports its own copy of simulate, has its own global state.
    return run_scenarios(positions, count, seed)


def run_subinterpreters(
    positions: list[Position], scenarios: int, workers: int, seed: int
) -> list[float]:
    sizes = _chunk_sizes(scenarios, workers)
    with InterpreterPoolExecutor(max_workers=workers) as pool:
        futures = [
            pool.submit(_interp_worker, positions, count, seed + i)
            for i, count in enumerate(sizes)
        ]
        return [pnl for future in futures for pnl in future.result()]
'''

print("InterpreterPoolExecutor pattern:")
print(SUB_INTERP_PATTERN)

InterpreterPoolExecutor pattern:

from concurrent.futures import InterpreterPoolExecutor

def _interp_worker(
    positions: list[Position], count: int, seed: int
) -> list[float]:
    # This function runs in its own sub-interpreter.
    # It imports its own copy of simulate, has its own global state.
    return run_scenarios(positions, count, seed)


def run_subinterpreters(
    positions: list[Position], scenarios: int, workers: int, seed: int
) -> list[float]:
    sizes = _chunk_sizes(scenarios, workers)
    with InterpreterPoolExecutor(max_workers=workers) as pool:
        futures = [
            pool.submit(_interp_worker, positions, count, seed + i)
            for i, count in enumerate(sizes)
        ]
        return [pnl for future in futures for pnl in future.result()]



## 4.5 Isolation and Data Races

A race requires two threads mutating **the same object**. Sub-interpreters cannot see each other's objects — the race simply cannot be written.

Data crosses the boundary only through **explicit channels**:
- `queue.SimpleQueue` — safe crossing point
- Pickling — most Python objects, serialised on the caller side
- Shareable types — a small set that crosses directly (bytes, int, float, ...)

**Safety by construction, not by discipline.**

## 4.6 Costs of Isolation

| Cost | Details |
|------|---------|
| Per-worker imports | Each interpreter re-imports your modules |
| Startup memory | Module copies multiplied by worker count |
| Pickle overhead | Most values crossing the boundary are pickled |
| Worker callables | Must be defined in an importable module, not `__main__` |

## 4.7 How the Callable Crosses the Boundary

The pool pickles the callable by **module path + name**, then each worker interpreter re-imports the module and looks up the name.

**Important gotcha:** A sub-interpreter's `__main__` is a blank module — the entry script never re-runs. If your worker function is defined in `__main__`, it fails to unpickle in the worker.

```python
# ❌ FAILS — defined in __main__
if __name__ == "__main__":
    def my_worker(x): return x * 2
    with InterpreterPoolExecutor() as pool:
        pool.submit(my_worker, 42)  # AttributeError: __main__ has no my_worker

# ✅ WORKS — defined in an importable module (e.g. mypackage.worker)
from mypackage.worker import my_worker
with InterpreterPoolExecutor() as pool:
    pool.submit(my_worker, 42)
```

## 4.8 Performance: Both Routes Deliver ~3×

Both parallelism approaches achieve similar scaling on the risk engine at 8 workers:

| Strategy | Build | ~Speedup at 8 workers |
|----------|-------|----------------------|
| Free-threading | `python3.14t` | ~3× |
| Sub-interpreters | `python3.14` (standard) | ~3× |

The Amdahl ceiling limits both: fan-out and reduce phases (sorting 40k P&Ls, merging results) stay serial, keeping the cap below 8×.

## 4.9 Decision Criteria

| Choose | When |
|--------|------|
| **Free-threading** | Hot shared data, lowest-latency handoffs, trusted payload |
| **Sub-interpreters** | Untrusted/stateful/fault-prone jobs, standard-build constraints |
| **asyncio (hybrid)** | IO fan-out at the front door — stays relevant for both |

A free-threaded process **can also host sub-interpreters** — one production API, swappable worker pool behind it.

## Module 4 — Key Takeaways

- Sub-interpreters give process-like isolation inside one OS process — no spawn, no OS IPC.
- Each interpreter holds its own GIL: 8 interpreters → 8 CPUs used — on the **standard** Python build.
- `InterpreterPoolExecutor` is the `concurrent.futures` drop-in (Python 3.14+).
- Worker callables must live in importable modules, not `__main__`.
- Isolation = safety by construction; free-threading = safety by discipline. Same ~3× speedup either way.

---
<a id='module-5'></a>
# Module 5 — Profiling Free-Threaded and Sub-Interpreter Workloads

## 5.1 Three Metrics to Track

| Metric | How to measure | What it tells you |
|--------|---------------|-------------------|
| **Throughput** | jobs/second at real concurrency | Overall system capacity |
| **Latency tail** | p95, p99 (not mean) | How bad the worst requests are |
| **Cores used** | `process_time() / perf_counter()` | Are threads actually running in parallel? |

The **cores-used ratio** is the definitive test:
- GIL threads: `cores_used ≈ 1.0` (all work serialised)
- Free threads or sub-interpreters: `cores_used ≈ N_workers`

In [12]:
import time
import threading


def cpu_burn(n: int) -> None:
    _ = sum(i * i for i in range(n))


def measure_cores(mode: str, workers: int, task_fn, n: int):
    cpu_start = time.process_time()
    wall_start = time.perf_counter()

    if mode == "single":
        task_fn(n)
    else:
        threads = [threading.Thread(target=task_fn, args=(n,)) for _ in range(workers)]
        for t in threads: t.start()
        for t in threads: t.join()

    cpu = time.process_time() - cpu_start
    wall = time.perf_counter() - wall_start
    cores = cpu / wall if wall > 0 else 0

    print(
        f"mode={mode:8s} workers={workers} "
        f"cpu={cpu:.3f}s wall={wall:.3f}s "
        f"cores_used={cores:.2f} "
        f"{'PARALLEL' if cores > 1.5 else 'serialised'}"
    )


N = 1_000_000
measure_cores("single", 1, cpu_burn, N)
measure_cores("threads", 4, cpu_burn, N)
print("\nOn standard GIL build: cores_used ≈ 1.0 for threads (GIL serialises).")
print("On free-threaded build: cores_used ≈ 4.0 for 4 threads.")

mode=single   workers=1 cpu=0.021s wall=0.021s cores_used=1.02 serialised
mode=threads  workers=4 cpu=0.083s wall=0.083s cores_used=1.00 serialised

On standard GIL build: cores_used ≈ 1.0 for threads (GIL serialises).
On free-threaded build: cores_used ≈ 4.0 for 4 threads.


## 5.2 Amdahl's Law

The theoretical speedup ceiling:

```
speedup = 1 / ((1 - p) + p / N)

Where:
  p = fraction of the run that parallelises
  N = number of workers
```

The risk engine's fan-out (running 40k scenarios in parallel) is nearly 100% parallelisable. The serial parts are fan-out setup and the final merge+sort. That's why 8 workers deliver ~3×, not 8×.

In [13]:
def amdahl_speedup(p: float, N: int) -> float:
    """Speedup according to Amdahl's law. p = parallel fraction, N = workers."""
    return 1 / ((1 - p) + p / N)


print(f"{'Workers':>8} {'p=0.95':>10} {'p=0.99':>10} {'p=1.00':>10}")
for N in [1, 2, 4, 8, 16, 32]:
    print(
        f"{N:>8}"
        f"{amdahl_speedup(0.95, N):>10.2f}x"
        f"{amdahl_speedup(0.99, N):>10.2f}x"
        f"{amdahl_speedup(1.00, N):>10.2f}x"
    )
print("\nEven 1% serial work caps 32-thread speedup at 24×.")

 Workers     p=0.95     p=0.99     p=1.00
       1      1.00x      1.00x      1.00x
       2      1.90x      1.98x      2.00x
       4      3.48x      3.88x      4.00x
       8      5.93x      7.48x      8.00x
      16      9.14x     13.91x     16.00x
      32     12.55x     24.43x     32.00x

Even 1% serial work caps 32-thread speedup at 24×.


## 5.3 The Four Profiling Tools

| Tool | Use for | Pitfall |
|------|---------|--------|
| `timeit.repeat` | Size the single-thread baseline | Warm up first — first run pays import + allocator |
| `cProfile` | Find the function-level hotspot | Inflates time ~3-4× due to instrumentation overhead |
| Stack sampler | Low-overhead confirmation of hotspot | Sampling rate varies; statistical, not exact |
| `process_time / perf_counter` | Verify cores are actually used | Wall-time alone cannot distinguish parallel from lucky |

**Rule:** rank hotspots with deterministic profiling (cProfile), but size the actual speedup with plain timing — never quote profiled seconds as performance.

In [14]:
# Stack sampler pattern from Module 5 (risk_engine/profiling.py)
import sys
import threading
import time
from collections import Counter


class StackSampler:
    """Lightweight sampling profiler — runs in a daemon thread."""

    def __init__(self, target_ident: int, interval: float = 0.001):
        self.target_ident = target_ident
        self.interval = interval
        self.counts: Counter[str] = Counter()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self) -> None:
        while not self._stop.is_set():
            time.sleep(self.interval)
            frame = sys._current_frames().get(self.target_ident)
            if frame is not None:
                self.counts[frame.f_code.co_name] += 1

    def __enter__(self):
        self._thread.start()
        return self

    def __exit__(self, *_):
        self._stop.set()
        self._thread.join()


def workload():
    """A fake workload to profile."""
    total = 0
    for i in range(2_000_000):
        total += i * i
    return total


sampler = StackSampler(threading.get_ident(), interval=0.001)
start = time.perf_counter()
with sampler:
    workload()
wall = time.perf_counter() - start

total_samples = sum(sampler.counts.values())
print(f"wall={wall:.3f}s  samples={total_samples}")
for name, count in sampler.counts.most_common(5):
    print(f"  {name:<30} {count:>4} samples  {100 * count / max(total_samples, 1):.1f}%")

wall=0.042s  samples=6
  workload                          5 samples  83.3%
  join                              1 samples  16.7%


## 5.4 Five Benchmark Traps (and How to Avoid Them)

| Trap | Symptom | Fix |
|------|---------|-----|
| **1. Single-thread overhead** | Reporting free-threaded speedup without showing the overhead row | Always report both denominators |
| **2. Unrepresentative inputs** | Tiny workloads make spawn/import dominate — everything looks bad | Pin input size across the whole course |
| **3. Skipping warm-up** | First run pays import, allocator growth, cold caches | Warm up then take best-of-N |
| **4. Believing one number** | Wall time alone cannot tell parallel from lucky | Pair wall time with `cores_used` |
| **5. Measuring through cProfile** | cProfile inflates time ~3.7× | Use `timeit` for sizing, `cProfile` only for ranking |

### The honest benchmarking method
1. Fixed, representative workload (40,000 scenarios throughout the course)
2. Warm-up, then best-of-N
3. Report: wall time + cores used + throughput per strategy — same table every time
4. See the speedup, then defend it

## 5.5 Comparing the Four Strategies

Full comparison requires two processes (because GIL is fixed at interpreter startup):

```
# Process A: python3.14 (standard build)
mode=single        wall=1.41s  cores=1.0
mode=processes×8   wall=0.42s  cores=7.8  speedup=3.4×
mode=interpreters×8 wall=0.41s cores=7.5  speedup=3.4×

# Process B: python3.14t (free-threaded build)
mode=single        wall=1.70s  cores=1.0  (20% overhead)
mode=threads×8     wall=0.46s  cores=7.2  speedup=3.7× (vs free-threaded baseline)
```

**Free-threading and sub-interpreters both achieve ~3× at 8 workers, with the same VaR-95 result** — the math is correct and the parallelism is genuine.

## Module 5 — Key Takeaways

- Track three metrics: throughput, latency tail (p99), and **cores used** (`process_time / perf_counter`).
- Amdahl's law caps speedup — even a small serial fraction limits scaling significantly.
- Use `cProfile` to rank hotspots, `timeit` (best-of-N with warm-up) to size performance.
- Never quote profiled time as actual performance — profilers inflate wall time 3–4×.
- Comparing free-threaded vs standard requires **two separate processes** (GIL fixed at startup).

---
<a id='module-6'></a>
# Module 6 — Shipping Free-Threaded Python to Production

## 6.1 The Production Architecture

The fully-shipped Globomantics risk engine combines all previous techniques:

```
Client
  │
  ▼
asyncio event loop  ── owns sockets, admission queue
  │
  ▼  (bounded asyncio.Queue for back-pressure)
Worker pool  ── ThreadPoolExecutor (--pool threads)   on free-threaded build
             └─ InterpreterPoolExecutor (--pool interpreters)  on standard build
  │
  ▼
Results store  ── plain dict, mutated only on the event-loop thread
```

The `--pool` flag is a **runtime flag, never a rewrite**. The same code serves both worker types.

In [15]:
# Key patterns from production.py (Module 6)
# Shown as a study cell — full version requires asyncio server

PROD_ENGINE_PATTERN = '''
from concurrent.futures import InterpreterPoolExecutor, ThreadPoolExecutor

POOLS = {
    "threads":      ThreadPoolExecutor,       # --pool threads  (free-threaded build)
    "interpreters": InterpreterPoolExecutor,  # --pool interpreters  (standard build)
}

class Engine:
    def __init__(self, pool: str, workers: int, queue_size: int):
        self.queue = asyncio.Queue(maxsize=queue_size)  # bounded = back-pressure
        self.executor = POOLS[pool](max_workers=workers)

    async def worker_loop(self):
        loop = asyncio.get_running_loop()
        while True:
            job_id = await self.queue.get()     # blocks until work available
            result = await loop.run_in_executor(
                self.executor, simulate_job, ...
            )                                   # offloads CPU, event loop stays free
            # update results dict on event loop — no locking needed

    def submit(self, scenarios, seed) -> tuple[str, dict]:
        if self.queue.full():
            return "503 Service Unavailable", {"error": "queue full"}  # back-pressure
        self.queue.put_nowait(job_id)
        return "202 Accepted", {"job_id": job_id}
'''

print(PROD_ENGINE_PATTERN)


from concurrent.futures import InterpreterPoolExecutor, ThreadPoolExecutor

POOLS = {
    "threads":      ThreadPoolExecutor,       # --pool threads  (free-threaded build)
    "interpreters": InterpreterPoolExecutor,  # --pool interpreters  (standard build)
}

class Engine:
    def __init__(self, pool: str, workers: int, queue_size: int):
        self.queue = asyncio.Queue(maxsize=queue_size)  # bounded = back-pressure
        self.executor = POOLS[pool](max_workers=workers)

    async def worker_loop(self):
        loop = asyncio.get_running_loop()
        while True:
            job_id = await self.queue.get()     # blocks until work available
            result = await loop.run_in_executor(
                self.executor, simulate_job, ...
            )                                   # offloads CPU, event loop stays free
            # update results dict on event loop — no locking needed

    def submit(self, scenarios, seed) -> tuple[str, dict]:
        if self.queue.full():
   

## 6.2 Back-Pressure with a Bounded Queue

| Queue type | Behaviour under overload |
|------------|-------------------------|
| **Unbounded** | Never says no — just grows. Overload becomes memory growth and silent latency tail. |
| **Bounded** | Full → immediate `503 Service Unavailable`. Client retries with intent. |

**Bounded queue + fixed worker pool = stated capacity.**

```
queue_size = 8, workers = 4
├── 4 jobs running (in workers)
├── 4 jobs queued (buffer)
└── job 9 → immediate 503, client knows to retry
```

Benefits:
- Overload sheds cleanly at the limit, no collapse
- Queue absorbs bursts; pool sets sustained throughput
- Expose `queue_depth` and `in_flight` to monitoring

In [16]:
import asyncio


async def demo_back_pressure():
    queue: asyncio.Queue[int] = asyncio.Queue(maxsize=3)  # bounded at 3

    # Simulate submitting 5 jobs
    for i in range(5):
        if queue.full():
            print(f"Job {i}: 503 Service Unavailable (queue full, depth={queue.qsize()})")
        else:
            queue.put_nowait(i)
            print(f"Job {i}: 202 Accepted (queue depth={queue.qsize()})")

    print(f"\nFinal queue depth: {queue.qsize()}")


await demo_back_pressure()

Job 0: 202 Accepted (queue depth=1)
Job 1: 202 Accepted (queue depth=2)
Job 2: 202 Accepted (queue depth=3)
Job 3: 503 Service Unavailable (queue full, depth=3)
Job 4: 503 Service Unavailable (queue full, depth=3)

Final queue depth: 3


## 6.3 Failure Containment

| Feature | Benefit |
|---------|--------|
| Per-job timeouts | Protects the caller's latency budget; no unbounded wait |
| A crashing job dies alone inside its worker | Sub-interpreter failure is isolated; other jobs continue |
| Isolation faults surface as catchable exceptions | Worker crash raises `concurrent.futures.BrokenExecutor` |
| Shutdown drains in-flight work | No dropped jobs on SIGTERM — accepted work finishes |

**Test it:** wire the engine end-to-end, then break it on purpose — inject timeouts, crash workers, flood the queue.

## 6.4 The Four Production Migration Gates

### Gate 1: C-extension audit
```bash
# Check PyPI for cp314t wheels:
pip index versions --python-version 3.14 mypackage

# At startup: wire a GIL-status probe
assert not sys._is_gil_enabled(), "GIL re-enabled — incompatible extension loaded"
```

### Gate 2: Dual-build CI
```yaml
# GitHub Actions matrix (conceptual)
strategy:
  matrix:
    python: ["3.14", "3.14t"]
steps:
  - run: pytest --fail-fast  # stress lane makes the gate mechanical
```

### Gate 3: Feature-flagged free-threading
```bash
# Instant rollback — no rebuild required
PYTHON_GIL=1 python3.14t server.py   # re-enables GIL at runtime
# or
python3.14t -X gil=1 server.py
```
Deploy as `--pool threads` on the free-threaded build. Rollback is changing an env var, not a binary.

### Gate 4: Benchmarking gates in CI
```
fixed workload + warm-up → wall, cores-used, throughput
→ a regression in throughput blocks the rollout like a failing test
```

In [17]:
# Service startup check — wire this into your application's startup sequence
import sys
import sysconfig


def assert_free_threaded() -> None:
    """Fail fast at startup if the GIL was unexpectedly re-enabled."""
    build = "free-threaded" if sysconfig.get_config_var("Py_GIL_DISABLED") else "standard"
    gil_on = sys._is_gil_enabled()
    print(f"Build: {build}  GIL enabled: {gil_on}")
    if build == "free-threaded" and gil_on:
        raise RuntimeError(
            "GIL was re-enabled — check C extension compatibility. "
            "Run with PYTHONWARNINGS=error to see which extension triggered it."
        )
    print("GIL status: OK")


assert_free_threaded()

Build: standard  GIL enabled: True
GIL status: OK


## 6.5 Observability: The /metrics Endpoint

The production engine exposes a `/metrics` endpoint that answers the questions operations needs:

```json
{
  "pool": "interpreters",
  "workers": 4,
  "queue_depth": 2,
  "in_flight": 3,
  "jobs_done": 1482,
  "jobs_rejected": 12,
  "throughput_busy": 14.3,
  "cores_used_busy": 3.8,
  "uptime_seconds": 103.6
}
```

- `throughput_busy` — jobs/s while at least one worker was active
- `cores_used_busy` — CPU/wall during busy periods: is parallelism real?
- `queue_depth` + `in_flight` — is the queue filling? Are workers saturated?

In [18]:
# BusyMeter: tracks CPU and wall time only while workers are active
# From: 06/demos/2. Assembling the production risk engine/risk_engine/production.py

import time


class BusyMeter:
    """Tracks CPU and wall time only during periods when at least one job is running."""

    def __init__(self) -> None:
        self.in_flight = 0
        self._busy_cpu = 0.0
        self._busy_wall = 0.0
        self._cpu_mark = 0.0
        self._wall_mark = 0.0

    def job_started(self) -> None:
        if self.in_flight == 0:
            self._cpu_mark = time.process_time()
            self._wall_mark = time.perf_counter()
        self.in_flight += 1

    def job_finished(self) -> None:
        self.in_flight -= 1
        if self.in_flight == 0:
            self._busy_cpu += time.process_time() - self._cpu_mark
            self._busy_wall += time.perf_counter() - self._wall_mark

    def busy_totals(self) -> tuple[float, float]:
        cpu, wall = self._busy_cpu, self._busy_wall
        if self.in_flight:
            cpu += time.process_time() - self._cpu_mark
            wall += time.perf_counter() - self._wall_mark
        return cpu, wall


# Demonstration
meter = BusyMeter()
meter.job_started()
time.sleep(0.1)           # simulate job running
meter.job_finished()

cpu, wall = meter.busy_totals()
print(f"Busy wall: {wall:.3f}s")
print(f"Busy CPU : {cpu:.3f}s")
print(f"Cores used during busy period: {cpu/wall:.2f}")

Busy wall: 0.103s
Busy CPU : 0.003s
Cores used during busy period: 0.03


## 6.6 Closing the Loop on the Baseline

Final performance summary of the Globomantics Risk Engine across the course:

```
Baseline (single-threaded, standard build):   ~1.41s  — cores_used=1.0

Free-threaded build, 8 threads:               ~0.38s  — cores_used≈7.2  → 3.7×
Standard build, 8 sub-interpreters:           ~0.41s  — cores_used≈7.5  → 3.4×

VaR-95 fingerprint stays identical: 71,618.80
```

**"The lock didn't get faster, it got optional."**

The key architectural insight: the hot path (Monte Carlo scenario loop) **shares nothing** across workers. When you can prove that, you get near-linear scaling with zero code changes to the inner loop.

## Module 6 — Key Takeaways

- Production architecture: `asyncio` event loop → bounded queue → worker pool (threads or interpreters).
- Bounded queue = stated capacity + immediate 503 under overload — graceful degradation.
- `--pool` is a runtime flag; instant rollback with `PYTHON_GIL=1` — no rebuild needed.
- Wire a GIL-status probe into service startup to catch silently re-enabled GIL from extensions.
- Four gates: C-extension audit → dual-build CI → feature-flagged rollout → benchmarking gates.

---
# Course Summary — The Complete Picture

## Architecture Decision Tree

```
Your workload is...

IO-bound (network, DB, files)?
  └─► asyncio  (one thread, event loop — no GIL friction)

CPU-bound AND...
  ├── hot shared data, trusted callers, lowest latency?
  │     └─► Free-threading (python3.14t)
  ├── isolated/untrusted jobs, standard-build constraint?
  │     └─► Sub-interpreters (python3.14, InterpreterPoolExecutor)
  └── existing multiprocessing code, no code changes allowed?
        └─► ProcessPoolExecutor (always worked, but higher overhead)

IO fan-out + CPU work?
  └─► asyncio front door + worker pool back end (hybrid boundary)
```

## Key Numbers to Remember

| Metric | Value |
|--------|-------|
| GIL switch interval | ~5 ms |
| Free-thread single-thread overhead | ~20% |
| 8-thread speedup (free-threaded / sub-interp) | ~3.5× |
| Process spawn time | ~300 ms |
| cProfile overhead | ~3–4× wall inflation |

## The Three Rules of Parallel Python

1. **Prove the hot path shares nothing** — if it does, scaling is trivial.
2. **Measure cores-used, not just wall time** — cores=1.0 means you're not actually parallel.
3. **Treat the interpreter switch like a dependency change** — dual-build CI, canary traffic, instant rollback.